In [ ]:

# ============================================================
# 1. IMPORTS
# ============================================================

import os
import getpass

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_ollama import ChatOllama, OllamaEmbeddings

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import (
    RunnableParallel,
    RunnableLambda,
    RunnablePassthrough
)


# ============================================================
# 2. CONFIGURE LANGSMITH OBSERVABILITY
# ============================================================

os.environ["LANGSMITH_TRACING"] = "true"

os.environ["LANGSMITH_API_KEY"] = getpass.getpass(
    "Enter your LangSmith API key: "
)

os.environ["LANGSMITH_PROJECT"] = "My-RAG-Observability"


# ============================================================
# 3. LOAD DOCUMENTS
# ============================================================

loader_fruits = TextLoader("./notes/fruits_and_vegetables.txt")
documents_fruits = loader_fruits.load()

loader_diet = TextLoader("./notes/human_nutrition_and_dietary_requirements.txt")
documents_diet = loader_diet.load()


# ============================================================
# 4. SPLIT DOCUMENTS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks_fruits = text_splitter.split_documents(documents_fruits)
chunks_diet = text_splitter.split_documents(documents_diet)


# ============================================================
# 5. CREATE EMBEDDINGS AND FAISS VECTOR STORES
# ============================================================

embedding_model = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectorstore_fruits = FAISS.from_documents(
    chunks_fruits,
    embedding_model
)

vectorstore_diet = FAISS.from_documents(
    chunks_diet,
    embedding_model
)


# ============================================================
# 6. CREATE RETRIEVERS
# ============================================================

fruits_retriever = vectorstore_fruits.as_retriever(
    search_kwargs={"k": 2}
)

nutrition_retriever = vectorstore_diet.as_retriever(
    search_kwargs={"k": 2}
)


# ============================================================
# 7. INITIALIZE CHAT MODEL
# ============================================================

llm = ChatOllama(
    model="qwen2.5:1.5b",
    temperature=0
)


# ============================================================
# 8. CREATE RETRIEVAL CHAIN
# ============================================================

# Run both retrievers
retrieval_chain = RunnableParallel(
    fruits=fruits_retriever,
    nutrition=nutrition_retriever
)


# Combine retrieved page_content into one context
def format_context(result):

    all_docs = result["fruits"] + result["nutrition"]

    context = "\n\n".join(
        doc.page_content for doc in all_docs
    )

    return context


retrieval_chain = (
    retrieval_chain
    | RunnableLambda(format_context)
)


# ============================================================
# 9. CREATE FINAL RAG PROMPT
# ============================================================

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the following context.

Context:
{context}

Question:
{question}

Answer:
""")


# ============================================================
# 10. CREATE COMPLETE RAG CHAIN
# ============================================================

rag_chain = (
    RunnableParallel(
        context=retrieval_chain,
        question=RunnablePassthrough()
    )
    | prompt
    | llm
    | StrOutputParser()
)


# ============================================================
# 11. INVOKE RAG WITH LANGSMITH TRACING
# ============================================================

question = "What nutrients do apples provide?"

response = rag_chain.invoke(
    question,
    config={
        "run_name": "Fruits-Nutrition-RAG",
        "tags": ["RAG", "FAISS", "Ollama"],
        "metadata": {
            "model": "qwen2.5:1.5b",
            "embedding_model": "nomic-embed-text"
        }
    }
)


# ============================================================
# 12. PRINT FINAL ANSWER
# ============================================================

print("\nFinal Answer:")
print(response)

print("\nExecution completed. Check your LangSmith project.")


C:\Users\Yoga\AppData\Local\Temp\ipykernel_12184\2778783040.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
